In [1]:
# !pip install flask numpy tensorflow pillow

In [ ]:
from flask import Flask, request, jsonify
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import io
from PIL import Image

app = Flask(__name__)
model = load_model("mobilenetV2_apple_disease_classifier.keras")

CLASS_NAMES = ['alternaria_leaf_spot', 'black_rot', 'cedar_apple', 'healthy', 'mosaic', 'powdery_mildew', 'scab']

def preprocess(img):
    img = img.resize((224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0
    return img_array

@app.route("/pests", methods=["POST"])
def predict():
    file = request.files['file']
    img = Image.open(io.BytesIO(file.read()))
    img_array = preprocess(img)
    
    predictions = model.predict(img_array)
    predicted_class = np.argmax(predictions[0])
    confidence = float(np.max(predictions[0]))

    return jsonify({
        "class_name": CLASS_NAMES[predicted_class],
        "confidence": confidence
    })

if __name__ == "__main__":
    app.run(port=5000, debug=False, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


c:\Users\harsh\TY\MIni_project_imp\venev\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 6 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


127.0.0.1 - - [05/Apr/2026 10:30:08] "POST /pests HTTP/1.1" 200 -
